# Session 1 — Can this gene be switched on?

**Applications of Machine Learning in Biotech and Medtech** · P-ITSZT-0061

---

An *Escherichia coli* genome assembly from an ICU patient contains a Shiga
toxin gene, *stx2*. Finding the gene tells you it is **there**. It does not
establish whether it is **being expressed**.

Transcription begins at a **promoter**: a stretch of DNA recognised by the
transcription machinery. A promoter can sit upstream of one gene or upstream of
an operon containing several genes. Recognising a promoter-like sequence is one
piece of the puzzle; regulation, genomic context, and the state of the cell also
matter. For *stx2*, phage regulation is part of that context.

Today you will build a model that reads 81 letters of DNA and returns a number
for how promoter-like they are. Our positive examples come from *E. coli*
K-12 MG1655, a laboratory reference strain; they are not patient isolates or a
collection specifically of *stx2* promoters. You will get a working classifier,
then investigate what its score actually measures. **This classifier does not
establish toxin expression or a patient's clinical risk.**

**By the end of this notebook you will have:**

1. A trained classifier and a score
2. A figure showing what the model decided mattered
3. A controlled comparison showing how the choice of negatives changes that score
4. At least one honest reason not to trust the score outside its benchmark


## How to use this notebook

**During class, run the notebook section by section in Google Colab.** Use a
CPU runtime; no GPU or Hugging Face login is needed. Every code cell already
works. Nothing needs to be filled in. Read the short explanation before each
cell, then run it and inspect the output. The experiments are designed for a
short CPU run; a cold package install or slow network can take longer.

**Pause when asked to predict.** In Section 10, run the three stages one at a
time, so you can respond to each result before seeing the next. Running all
cells at once is useful for checking that the notebook executes, but the
classroom exercise includes decisions made before the answers appear.

You will never be asked to write code from an empty cell in the main exercise.
An exercise looks like this:

> **Try it.** Change the k-mer length in the working experiment, run that cell
> again, and write down what happened to the score.

**Save your own copy:** File → Save a copy in Drive. That preserves your edits
and answers; the temporary Colab runtime can disappear when a session ends.

**Work in pairs.** One person types, the other decides what to type.
Swap every twenty minutes. Make a prediction before changing a setting, then
record what actually happened.


### Data and reuse

We use [neuralbioinfo/bacterial_promoters](https://huggingface.co/datasets/neuralbioinfo/bacterial_promoters),
distributed under **CC BY-NC 4.0**. The dataset was assembled using promoters
from the Prokaryotic Promoter Database. It supplies several negative classes:
generated random sequences, sampled coding DNA, and sequences generated using
a composition model.

The source documentation describes **75 organisms**. The full training table
contains **76 distinct non-missing species-name strings**. These are metadata
categories, including strain names and naming variants; 76 strings do not
establish 76 biological species. We use one explicitly selected category of
*E. coli* positives in today's main experiment.

Preserve the dataset attribution and non-commercial licence when reusing its
sequences.


## 1. Setup

The next cell installs `datasets`, the package that retrieves the public data,
and imports tools for tables and plots. Everything else is already available in
Colab. Normal installation output is hidden; errors remain visible.

`SEED = 42` controls random choices throughout the notebook. Keeping it fixed
makes comparisons easier to reproduce. Leave it alone on your first run; later
you will change seeds deliberately. Look for **Ready.** before continuing.


In [ ]:
# One install. Everything else is already in Colab.
!pip install datasets -q > /dev/null

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
Path("figures").mkdir(exist_ok=True)

# One seed, used throughout the notebook for reproducible random choices.
SEED = 42

print("Ready.")


The quiet installation is intentional. You do not need to understand the
imports yet: `pandas` handles tables, `numpy` handles numerical arrays, and
`matplotlib` plus `seaborn` draw the figures. You will see each one used below.

The full dataset contains about 247,500 short DNA sequences across its supplied
splits, each 81 letters long and labelled promoter or non-promoter. Its compressed
data is small enough for a classroom download. We use the supplied **training
split** for today's modelling experiments; other supplied splits remain
separate.


## 2. Load the data

This cell downloads the dataset and turns its **full training split** into a
table called `promoters_all`. We need the full split before selecting our
organism: taking a small random sample first would throw away most of the
*E. coli* positives we want to use.

If the Hub cannot be reached, the cell reports the error and uses the bundled
exercise pool. That pool contains real source rows, including every positive
in our selected *E. coli* category and the negative examples needed for the
default comparisons. The printed source tells you which route was used.

The checks after loading verify that the expected columns, 81-letter DNA
windows, and binary labels are present. A malformed dataset should produce a
clear error. Records with ambiguous DNA letters such as N, W, or Y are
explicitly excluded so today's k-mers use A/C/G/T only. Look for the source,
rows loaded, and number excluded. The bundled pool was prepared using the
same sequence checks.


In [ ]:
from datasets import load_dataset

# Reset this on each run so an old download cannot leak into the fallback path.
dataset = None

dataset = load_dataset("neuralbioinfo/bacterial_promoters",
                       revision="38f748a5ea68896c9467087e41c0210e5d5b5760")
promoters_all = dataset["train"].to_pandas()
print("Loaded from Hugging Face.")


# Validate after the fallback decision: bad data must not be silently replaced.
required_columns = [
    "segment_id", "ppd_original_SpeciesName", "Strand", "segment",
    "class_label", "L", "prom_class", "y",
]
missing_columns = set(required_columns) - set(promoters_all.columns)
if missing_columns:
    raise ValueError(f"Missing dataset columns: {sorted(missing_columns)}")
if promoters_all[["segment", "L", "prom_class", "y"]].isna().any().any():
    raise ValueError("Sequences, lengths, construction categories, and labels must not be missing.")
if not promoters_all["segment"].str.fullmatch("[ACGTRYSWKMBDHVN]{81}").all():
    raise ValueError("Every segment must contain exactly 81 valid DNA symbols.")
if not promoters_all["L"].eq(81).all():
    raise ValueError("The declared sequence lengths must also equal 81.")
if set(promoters_all["y"].unique()) != {0, 1}:
    raise ValueError("The labels must contain both 0 and 1, with no other values.")

print("Rows loaded:", len(promoters_all))
# Five source records contain ambiguous DNA letters such as N, W, or Y.
# Exclude these explicitly so each feature is a k-mer over A, C, G, and T.
canonical_dna = promoters_all["segment"].str.fullmatch("[ACGT]{81}")
print("Rows excluded for ambiguous DNA letters:", int((~canonical_dna).sum()))
promoters_all = promoters_all.loc[canonical_dna].reset_index(drop=True)

if dataset is None:
    print("Using tailored E. coli/negative pools, not the full training split.")
    print("Default SEED = 42 reproduces the online selection.")


### Narrowing to one organism, and one kind of negative

Two decisions before we build anything. Both change the question the model
will answer.

**One organism.** We select the exact species-name category for *Escherichia
coli* K-12 MG1655 from the **full training split**. At the pinned dataset
revision, it contains **6,801 promoter rows**. Two additional positive records
use an alternative spelling of the strain name; our explicit exact-string
filter leaves them outside this exercise. Inspect the selected name printed
by the next cell instead of guessing how a strain was recorded.

This is a well-studied *E. coli* reference strain, linking our classroom task
to the organism in the opening story. Restricting the organism reduces one
source of variation, but *E. coli* itself uses **multiple sigma factors**.
One organism does not mean one promoter motif family.

**One kind of negative.** For now, we use only `RND`: generated random
nucleotide sequences. The other negative classes remain available for
Section 10. There, changing the comparison will be the experiment.

**Balance the two labels.** The next cell keeps the selected positives and
draws the same number of RND negatives with `SEED`. For the default data, look
for **6,801 promoters + 6,801 negatives = 13,602 sequences**. There is no
20,000-row subsampling step. Later experiments change how many of these
training examples the classifier sees.


In [ ]:
SPECIES = "Escherichia coli str K-12 substr. MG1655"  # One exact source label.
NEGATIVE_CLASS = "RND"  # Random nucleotide negatives; Section 10 changes the comparison.

# Filter the FULL available source before sampling. Never filter the old 20k sample.
positive_pool = promoters_all[
    (promoters_all["y"] == 1)
    & (promoters_all["ppd_original_SpeciesName"] == SPECIES)
]
random_negative_pool = promoters_all[
    (promoters_all["y"] == 0) & (promoters_all["prom_class"] == NEGATIVE_CLASS)
]
n_per_class = min(len(positive_pool), len(random_negative_pool))
if n_per_class < 20:
    raise ValueError("Too few rows for the selected organism and negative class.")

# Sorting fixes row order after sampling, including when using the saved source pools.
positives = positive_pool.sample(n=n_per_class, random_state=SEED)
positives = positives.sort_values("segment_id").reset_index(drop=True)
negatives = random_negative_pool.sample(n=n_per_class, random_state=SEED)
negatives = negatives.sort_values("segment_id").reset_index(drop=True)
promoters = pd.concat([positives, negatives], ignore_index=True)
promoters = promoters.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Organism:  ", SPECIES)
print("Negatives: ", NEGATIVE_CLASS, "(random nucleotide sequence)")
print()
print("Promoters:    ", len(positives))
print("Non-promoters:", len(negatives))
print("Total:        ", len(promoters))


### One thing to notice about what we just did

We restricted the **promoters** to *E. coli*. We could not apply a matching
species filter to the **negatives**, because **all negative classes in the
source training table lack species-name metadata**.

For our current RND negatives, a biological species is not defined: these
sequences were generated. Later, `CDS` negatives come from real coding DNA
sampled from sequenced genomes, but this table does not identify their source
organisms. The composition-model negatives were also generated; they are not
documented here as a negative set specifically matched to our selected
*E. coli* positives.

So today's main model separates ***E. coli* promoter windows from generic
random sequence**. It may use properties of the organism or the construction
process, alongside any promoter-associated signal. This is a **species and
negative-source confound**: more than one explanation can produce a good score.

We have not removed the shortcut problem. We have made the task more specific.
Keep this limitation in mind when you look at Figure 4, and again when we
change the negative class in Section 10.


## 3. Look at the data

Before building anything, look at what you have. This is the step people skip
and then regret. In your Design Sheet, this fills **box 2: what is one sample?**
and **box 3: what is its label?**

The next cell displays five rows. Find the DNA sequence and the label attached
to it. A row is one short window of DNA, not a whole genome, a patient, or a gene.


In [ ]:
promoters.head()


### What is one row?

**One row is one 81-letter window of DNA and its associated label and metadata.**
The prediction task is binary classification: choose between the two dataset
labels. The model will receive the DNA sequence; the other descriptive columns
help us inspect the benchmark and its design.

| Column | Meaning |
|---|---|
| `segment_id` | Identifier for this sequence window |
| `ppd_original_SpeciesName` | Original species-name category, where provided; missing for all negative classes |
| `Strand` | Recorded DNA strand, `+` or `−`, where provided |
| `segment` | The 81-letter DNA sequence that the model reads |
| `class_label` | Text description of the assigned class |
| `L` | Recorded sequence length; we also checked the actual string length |
| `prom_class` | Provenance/construction category, including the positive category and different kinds of negative |
| `y` | Numeric target: `1` for a promoter, `0` for a negative example |

The next cell checks lengths and label balance, then prints the provenance
categories for the working set and for the full source training split.
The second view shows what we deliberately left out. In fallback mode, full
source counts are saved metadata; the available exercise pool is smaller.

Do not memorise category names: inspect the output. Remember that `prom_class`
includes the positive category as well as different negative categories.


In [ ]:
print("Sequence length:", promoters["segment"].str.len().unique())
print()
print("Label counts: 1 = E. coli promoter, 0 = random nucleotide example")
print(promoters["y"].value_counts())
print()
print("Categories used in today's baseline:")
print(promoters["prom_class"].value_counts())
print()
print("Full source training categories, after the A/C/G/T check:")
if dataset is not None:
    print(promoters_all["prom_class"].value_counts())
else:
    print(pd.Series(source_catalog["full_class_counts"]))
    print("Full-source counts come from the verified snapshot catalogue.")


Look at the two provenance tables again.

Our working set contains **promoters and RND negatives only**. The full source
table offers other comparisons. Those alternatives were deliberately set
aside; they have not disappeared from the question we need to ask.

The positives come from curated promoter records. The negative label combines
different routes to selecting or generating a comparison sequence. Its `0`
tells you how the benchmark was assembled; it does not prove that a sequence
could never support transcription under any condition. Experiments can test
for an absence of activity in particular conditions, but that is a different
claim.

Keep the alternative categories in mind. We will investigate their effect at
the end of the main block, after you have a model and a score to question.

First, the next cell prints three positive and three negative sequences with
their species-name metadata. Follow the two loops: the outer loop selects a
label; the inner loop prints examples. Look for any difference you can describe
before calculating anything.


In [ ]:
# Read three examples from each class, together with their recorded species.
for label_value in [1, 0]:
    examples = promoters[promoters["y"] == label_value].head(3)
    label_name = "promoter" if label_value == 1 else "negative example"
    print("Label", label_value, "(", label_name, ")")
    for _, row in examples.iterrows():
        species_name = row["ppd_original_SpeciesName"]
        if pd.isna(species_name):
            species_name = "species not recorded"
        print(" ", row["segment"], " |", str(species_name)[:28])
    print()


Can you reliably tell these sequences apart by eye? Probably not. A few familiar
letters do not settle the question. We need a representation that exposes
patterns across many labelled examples.

### Figure 1 — Start with one simple property

**GC fraction** is the fraction of letters that are G or C. We calculate it
for every sequence with an explicit loop, then compare **our selected *E. coli*
promoters with RND negatives** using overlaid histograms. Each histogram is
normalised to its own class size.

Look for a shift, overlap, or differences in spread. Do not assume the classes
will separate cleanly. This figure describes the working comparison; it is not
a general comparison between promoters and all other bacterial DNA. It is
displayed and saved as `figures/fig1_gc_content.png` at 150 dpi.


In [ ]:
# Count G and C letters and divide by the number of letters in the window.
gc_fractions = []
for sequence in promoters["segment"]:
    gc_count = sequence.count("G") + sequence.count("C")
    gc_fraction = gc_count / len(sequence)
    gc_fractions.append(gc_fraction)
promoters["gc_fraction"] = gc_fractions

figure, axis = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 31)
for label_value, label_name, colour in [
    (0, "Negative examples", "#476A9B"),
    (1, "Promoters", "#D57A32"),
]:
    class_gc = promoters.loc[promoters["y"] == label_value, "gc_fraction"]
    axis.hist(
        class_gc, bins=bins, density=True, alpha=0.55,
        label=label_name, color=colour, edgecolor="white", linewidth=0.4,
    )
axis.set(xlabel="GC fraction", ylabel="Density", title="DNA composition by dataset label")
axis.legend()
figure.tight_layout()
figure.savefig("figures/fig1_gc_content.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(figure)

print("Mean GC fraction by label:")
print(promoters.groupby("y")["gc_fraction"].mean().round(3))


### Biology, and a possible shortcut

If promoter examples have a lower GC fraction here, they are more AT-rich on
average. Common bacterial σ70 promoter motifs contain many A and T letters,
and local DNA opening is part of transcription initiation. Duplex stability
depends on neighbouring bases as well as base pairing; hydrogen-bond counts
alone do not explain promoter function. Our *E. coli* positive set includes
more than one sigma-factor class, so this explanation is not a motif annotation
for every row.

The important modelling question is this: **could a model score well by
learning overall composition instead of recognising promoter structure?**
These are different explanations, and a score alone cannot distinguish them.
If the distributions largely overlap, GC fraction alone will be less useful;
more detailed sequence patterns may still matter. Compare what the figure
actually shows with your expectation.

The next cell confirms our organism filter and the missing negative metadata.
Look for one recorded species-name category among the working positives and
no recorded species among the working negatives. This is a check of the data
selection we intended, not evidence that the negatives came from *E. coli*.


In [ ]:
print("Positive organism:", positives["ppd_original_SpeciesName"].unique())
print("Distinct positive organism-name categories:", positives["ppd_original_SpeciesName"].nunique())
print("Negatives without a species label:", negatives["ppd_original_SpeciesName"].isna().sum())
print("Negative examples in this comparison:", len(negatives))


In [ ]:
#sorted([str(la) for la in list(promoters_all["ppd_original_SpeciesName"].unique())])

Hold onto that distinction for **Session 4**, when we ask whether a model can
generalise to organisms it has never seen.

The full training table has **76 distinct non-missing species-name strings**;
the dataset documentation describes **75 organisms**. Strain names and aliases
mean that a count of strings is not automatically a count of biological
species. Our working positive set deliberately keeps one exact category.

Neither that filter nor a random train/test split establishes performance on
an unseen species. The missing organism provenance for negatives will also
need explicit attention in any future attempt at an organism-based split.


## 4. See the biology 

Promoter structure depends on **where letters occur**, not just how many times
they occur. In aligned promoter windows we can ask: *what fraction of sequences
has A, C, G, or T at each position?*

For this illustration, we retain the **864 positive examples in the supplied
`test_sigma70` split** as a cleaner reference set. Comparing the alternatives
during notebook development showed a sharper positional pattern here than
across our selected *E. coli* training positives. The latter include promoters
recognised by multiple sigma factors: selecting one organism does not turn
them into a single motif family.

**Figure 2 describes this sigma70 reference, not the main training data.** It
is a biological illustration, not a model evaluation. These reference rows do
not enter today's training experiments. We inspect the split here and do not
later claim it as an untouched final test.

The next cell selects the reference positives from the downloaded dataset or
the bundled copy. Look for the printed source and **864 positives** before
interpreting the heatmap.


In [ ]:
# Use a clearly labelled, cleaner sigma70 reference, not the training positives.
sigma70 = dataset["test_sigma70"].to_pandas()
sigma70_positives = sigma70.loc[sigma70["y"] == 1].copy()
motif_source = "sigma70 reference (not the training set)"
if not sigma70_positives["segment"].str.fullmatch("[ACGT]{81}").all():
    raise ValueError("The reference heatmap needs 81-letter A/C/G/T sequences.")
print("Illustration source:", motif_source)
print("Reference positives:", len(sigma70_positives))
print("E. coli training-pool positives:", len(positives))


### Figure 2 — A pattern from counting letters

We build a **4 × 81 table from the sigma70 reference positives**. Each entry
is the fraction of sequences containing one base at one position. The loops
make the calculation visible: choose a base, choose a position, count matching
sequences, divide by the total.

The heatmap has one row per base and one column per raw sequence index,
**0 through 80**. Brighter cells mean that base appears more often at that
position. Each vertical column should add to 1 across the four bases.

We retain raw indices instead of assuming a verified transcription-start-site
offset. An AT-rich pattern consistent with a −10 element may occur roughly
two-thirds of the way along these windows, but identifying its exact biological
coordinates requires checking the source extraction convention. This is a
reference illustration; it does not imply that every main-path positive has
the same pattern. The figure is displayed and saved as
`figures/fig2_position_frequency.png` at 150 dpi.


In [ ]:
bases = ["A", "C", "G", "T"]
sequence_length = 81
position_frequencies = np.zeros((len(bases), sequence_length))

# Each row is one DNA base; each column is one aligned position in the window.
for base_index, base in enumerate(bases):
    for position in range(sequence_length):
        matching_sequences = 0
        for sequence in sigma70_positives["segment"]:
            if sequence[position] == base:
                matching_sequences += 1
        position_frequencies[base_index, position] = (
            matching_sequences / len(sigma70_positives)
        )

figure, axis = plt.subplots(figsize=(12, 3.5))
sns.heatmap(
    position_frequencies, ax=axis, cmap="viridis", vmin=0, vmax=1,
    yticklabels=bases, xticklabels=False,
    cbar_kws={"label": "Base frequency"},
)
# Label every tenth raw index; 81 tiny tick labels would be hard to read.
tick_indices = np.arange(0, sequence_length, 10)
axis.set_xticks(tick_indices + 0.5)
axis.set_xticklabels(tick_indices, rotation=0)
axis.set_yticklabels(bases, rotation=0)
axis.set(
    xlabel="Raw sequence index (0–80)", ylabel="Base",
    title="Base frequency by position — " + motif_source,
)
figure.tight_layout()
figure.savefig("figures/fig2_position_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(figure)


For many bacterial **σ70 promoters**, the −10 element is called the
**Pribnow box**, with consensus `TATAAT`. A second familiar element near −35 has
consensus `TTGACA`. The numbers are positions relative to the transcription
start site, not our raw array indices.

Look for positions enriched in A or T in Figure 2. In this sigma70 reference,
a concentrated pattern is consistent with promoter structure. A heatmap alone
does not prove the identity or coordinate of a motif. Our selected *E. coli*
training positives span multiple promoter types, so their average positional
pattern can be weaker or broader even though the organism is the same.

**No model has been trained yet.** Any visible pattern emerged from counting
letters across sequences. A suitable representation can preserve positional
information for a model to use. Next, we try a simpler representation and ask
what it keeps and what it loses.


### A tendency, not a fixed string

`TATAAT` is a **consensus**, a summary of common letters across many promoters.
Individual promoters can differ from it. Promoter recognition also depends on
other elements, their spacing, and which sigma factor is involved. Conversely,
finding the exact word `TATAAT` somewhere in a genome is not sufficient evidence
of a functional promoter.

That is why a plain text search for one word is an incomplete approach. We will
let a model combine evidence from many short sequence patterns, and then ask
whether its apparent success reflects the biology we care about.


## 5. What is a k-mer? 

Our classifier needs a numerical description of each DNA sequence. One simple
conversion is to **count short, overlapping substrings**. A *k-mer* is a
substring of length *k*.

For `ATGCATG` with **k = 3**, slide a three-letter window one position at a time:

| Start index | 3-mer |
|---|---|
| 0 | `ATG` |
| 1 | `TGC` |
| 2 | `GCA` |
| 3 | `CAT` |
| 4 | `ATG` |

The counts are `ATG: 2`, `TGC: 1`, `GCA: 1`, `CAT: 1`. Windows overlap: we do not
divide the sequence into non-overlapping blocks.

The next cell performs that same calculation on one real sequence. `counts`
is a dictionary that associates each word with its count. `counts.get(kmer, 0)`
means “use its current count, or zero if this is its first appearance.” Look at
the sequence, how many distinct words it contains, and a few example counts.


In [ ]:
example_sequence = promoters["segment"].iloc[0]
K_DEMO = 3

counts = {}
for start in range(len(example_sequence) - K_DEMO + 1):
    kmer = example_sequence[start : start + K_DEMO]
    counts[kmer] = counts.get(kmer, 0) + 1

print("Sequence:", example_sequence[:30], "...")
print("Number of distinct", K_DEMO, "-mers found:", len(counts))
print("First few:", dict(list(counts.items())[:6]))
print("Total overlapping windows:", sum(counts.values()))
# Try it later: change K_DEMO to 4 and compare distinct words and total windows.


That is all a k-mer counter does. Scikit-learn provides an efficient character
substring counter called `CountVectorizer`, so we use that from here. Its
`analyzer="char"` setting makes it count characters rather than words separated
by spaces; DNA is one uninterrupted string.

The representation changes, but nothing mysterious happens: each sequence
becomes a row of counts. A k-mer absent from that sequence gets a zero.


## 6. Turn sequences into numbers 

For the first model, choose **k = 4**. There are at most **4⁴ = 256 possible
4-mers** over A, C, G, and T. Each 81-letter sequence supplies 78 overlapping
4-mer windows, so many possible words will have a zero count in an individual
sequence.

The next cell sets the k-mer length and prints these two quantities. We will
split sequences into training and test sets **before fitting the counter** in
the next section. That lets the training data define the vocabulary; the test
data is transformed using the same vocabulary.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

K = 4  # The working k-mer length. Change it in the controlled experiment later.

print("Possible", K, "-mers over A/C/G/T:", 4 ** K)
print("Overlapping windows in an 81-letter sequence:", 81 - K + 1)


The **feature matrix** will have one row per sequence and one column per k-mer
found in the training vocabulary. A cell contains that word's count in that
sequence. The **labels** are the known promoter/negative assignments used to
train and assess the classifier; they are not inputs when making a prediction
for a new sequence.

We call these `features` and `labels`. Most tutorials and much of scikit-learn's
documentation call them `X` and `y`. Same things. The dataset's existing `y`
column is the source of our `labels`.


### What counting discards · Design Sheet box 4

Counting throws something away: **absolute position**.

A particular k-mer appearing near the start of the window contributes the same
count as that k-mer appearing near the end. With k = 6, for example, `TATAAT`
gets the same feature count wherever it occurs. With k = 4, we count its shorter
fragments instead. Either way, the representation does not record the start
position of each occurrence.

Figure 2 asked *where* letters are enriched. Promoter biology also depends on
positions and spacing. A bag of k-mer counts preserves local words but loses
their absolute locations and much of their arrangement.

We choose this representation today because it is simple and fast. It is the
first rung of a ladder we climb throughout the course. In **Design Sheet
box 4**, write both what your representation retains and what it discards.


## 7 — Train a model 

We hold back **25%** of the balanced *E. coli* promoter/random-negative set.
The model learns from the other 75%; its score is calculated on held-out rows.
`stratify` keeps the two labels similarly represented on each side. We split
**before** learning the k-mer vocabulary: `fit_transform` learns columns on
training DNA, while `transform` uses those same columns for test DNA.

This is a **random split within this comparison**. Related sequences can occur
on both sides, and all positive examples come from the chosen organism label.
It does not test transfer to other species, independence from related sequences,
or performance against real background DNA. Those questions remain for Session 4.

The next cell counts k-mers, fits logistic regression, and measures ROC-AUC.
`fit` learns weights; `predict_proba` produces scores. Look for a strong first
ranking score, but read the actual output rather than aiming for a promised
number. The printed matrix dimensions mean *sequences × k-mer columns*.

We call these `features` and `labels`; most tutorials call them `X` and `y`.
The first score answers a specific question: **can we distinguish these
E. coli promoter sequences from the selected random nucleotide sequences?**


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, RocCurveDisplay


# K was set in Section 6; keep that value for the baseline.
labels = promoters["y"].to_numpy()
row_indices = np.arange(len(promoters))
train_indices, test_indices = train_test_split(
    row_indices, test_size=0.25, random_state=SEED, stratify=labels
)
sequences_train = promoters["segment"].iloc[train_indices]
sequences_test = promoters["segment"].iloc[test_indices]
labels_train = labels[train_indices]
labels_test = labels[test_indices]

# Each column counts one DNA substring. The test set does not choose the columns.
counter = CountVectorizer(analyzer="char", ngram_range=(K, K), lowercase=False)
features_train = counter.fit_transform(sequences_train)
features_test = counter.transform(sequences_test)

# Logistic regression combines the counts into a promoter score.
# liblinear is a CPU solver suited to this small, sparse binary problem.
model = LogisticRegression(solver="liblinear", max_iter=2000, random_state=SEED)
model.fit(features_train, labels_train)
scores = model.predict_proba(features_test)[:, 1]
auc = roc_auc_score(labels_test, scores)
baseline_auc = auc

print("Training matrix shape:", features_train.shape)
print("Test sequences:", features_test.shape[0])
print("First few k-mers:", counter.get_feature_names_out()[:8])
print("ROC-AUC:", round(auc, 3))

# Run all resets the table. Rerunning later experiments adds new rows.
results = pd.DataFrame(columns=["experiment", "setting", "auc", "n_train", "n_test"])
results.loc[len(results)] = ["baseline", f"k={K}; E. coli vs RND", auc, len(labels_train), len(labels_test)]
display(results.round({"auc": 3}))


**You have trained a model. It reads DNA and returns a number.**

### What that number means

**ROC-AUC** is the probability that a randomly chosen labelled promoter receives
a higher score than a randomly chosen negative, with a tied pair counting as half.

- **0.5** — chance-level ranking, our reference line.
- **1.0** — perfect ranking: every positive scores above every negative.
- **Below 0.5** — reversed or worse-than-chance ranking is possible; 0.5 is not a floor.

AUC measures **ranking**, not the percentage of correct answers. It does not
choose a decision threshold, establish a calibrated probability of expression,
or tell us **what the negatives were**. Hold that last point.

The next cell draws the ROC curve. Each point uses a different score threshold;
thresholds are a Session 3 topic. Look at how far the curve rises above the
diagonal, and connect the area under it to the score you just saw.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
RocCurveDisplay.from_predictions(labels_test, scores, ax=ax, name=f"{K}-mer baseline")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Chance ranking")
ax.set(title=f"Promoter ranking | ROC-AUC = {baseline_auc:.3f}",
       xlabel="False positive rate", ylabel="True positive rate")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("figures/fig3_roc.png", dpi=150)
plt.show()
plt.close(fig)


## 8 — What did it learn? 

The model assigned a weight to every k-mer. A positive weight means that,
holding the other counts fixed, another copy pushes the score toward promoter;
a negative weight pushes it away. These are associations in this comparison,
not measurements of biological importance.

Look at the 20 most positive and 20 most negative weights. **Do the top words
look like fragments of `TATAAT` and `TTGACA`, simply AT-rich words, or some
other pattern?** They may also reflect the composition of the selected
*E. coli* positives relative to the global random-negative pool.

Compare the next figure with the labelled sigma70 reference, while remembering
that the reference is not exactly the training population. These weights
belong to the original RND baseline, even after later experiments.


In [ ]:
weights = pd.DataFrame({
    "kmer": counter.get_feature_names_out(),
    "weight": model.coef_[0],
})
top_weights = weights.nlargest(20, "weight").sort_values("weight")
bottom_weights = weights.nsmallest(20, "weight").sort_values("weight")

fig, axes = plt.subplots(1, 2, figsize=(10, 7))
axes[0].barh(bottom_weights["kmer"], bottom_weights["weight"], color="#d47d36")
axes[1].barh(top_weights["kmer"], top_weights["weight"], color="#167b88")
axes[0].set_title("Toward the negative label")
axes[1].set_title("Toward the promoter label")
for ax in axes:
    ax.set_xlabel("Learned coefficient")
    ax.axvline(0, color="black", linewidth=0.8)
fig.suptitle("What counts as promoter-like to this model?")
fig.tight_layout()
fig.savefig("figures/fig4_kmer_weights.png", dpi=150)
plt.show()
plt.close(fig)


Some high-weight words may resemble motif fragments, others general sequence
composition. Their positions are absent from the model's input. Neither the
weights nor the AUC proves that it recognised functional promoter structure.

The next cell measures one narrow aspect of that interpretation: the mean AT
fraction of the top 20 words compared with the full vocabulary. Read the
numbers from this run. The old multispecies result does not transfer to this
new *E. coli* comparison. Higher AT content supports an AT-enrichment observation;
a similar value does not rule out other compositional differences.

This diagnostic comes from the same fitted model, so it is another view of
the evidence rather than an independent proof of its mechanism. Write down
your interpretation and one observation that could challenge it.


In [ ]:
top_at_fractions = []
for kmer in top_weights["kmer"]:
    top_at_fractions.append((kmer.count("A") + kmer.count("T")) / len(kmer))

all_at_fractions = []
for kmer in weights["kmer"]:
    all_at_fractions.append((kmer.count("A") + kmer.count("T")) / len(kmer))

print("Mean AT fraction of top 20 k-mers:", round(np.mean(top_at_fractions), 3))
print("Mean AT fraction of all k-mers:   ", round(np.mean(all_at_fractions), 3))


## 9 — Change one thing at a time

Change one thing, rerun, record. Not two things. The growing `results` table is
your lab record; store the full score and round only for display.

### Experiment A — the length of a k-mer

We will try k = 2, 3, 4, 5, and 6 on the **same** training and test rows.
The next two cells deliberately repeat the full counting, fitting, and scoring
steps. Read them beside the baseline. The procedure is unchanged; only k changes.

First, run the complete 2-mer example. Look for a much narrower feature matrix
and record whether the AUC is higher or lower than the baseline.


In [ ]:
k_value = 2  # Try it: change only this value and record the score.
trial_counter = CountVectorizer(analyzer="char", ngram_range=(k_value, k_value), lowercase=False)
trial_features_train = trial_counter.fit_transform(sequences_train)
trial_features_test = trial_counter.transform(sequences_test)
trial_model = LogisticRegression(solver="liblinear", max_iter=2000, random_state=SEED)
trial_model.fit(trial_features_train, labels_train)
trial_scores = trial_model.predict_proba(trial_features_test)[:, 1]
trial_auc = roc_auc_score(labels_test, trial_scores)

print("Training matrix shape:", trial_features_train.shape)
results.loc[len(results)] = ["k-mer length", f"k={k_value}", trial_auc, len(labels_train), len(labels_test)]
display(results.round({"auc": 3}))


Now use 3-mers. This is the third visible example of counting, fitting, and
scoring. Compare its columns with the 2-mer example and check how much the
score changes. These rows stay in the same results table.


In [ ]:
k_value = 3  # Try it: keep everything else unchanged.
trial_counter = CountVectorizer(analyzer="char", ngram_range=(k_value, k_value), lowercase=False)
trial_features_train = trial_counter.fit_transform(sequences_train)
trial_features_test = trial_counter.transform(sequences_test)
trial_model = LogisticRegression(solver="liblinear", max_iter=2000, random_state=SEED)
trial_model.fit(trial_features_train, labels_train)
trial_scores = trial_model.predict_proba(trial_features_test)[:, 1]
trial_auc = roc_auc_score(labels_test, trial_scores)

print("Training matrix shape:", trial_features_train.shape)
results.loc[len(results)] = ["k-mer length", f"k={k_value}", trial_auc, len(labels_train), len(labels_test)]
display(results.round({"auc": 3}))


### Package the familiar steps

This function contains the same counting, fitting, and scoring code you just
ran, wrapped so we can call it repeatedly without copying. Nothing new happens
in those steps. You have now seen them explicitly three times.

The optional `n_train` takes a stratified subset **from the training side only**.
The test rows stay fixed for a fixed seed. This lets us change training size
without also changing the test. The function returns the AUC; it does not
overwrite the baseline model or add hidden results. The next cell defines it
and prints a readiness message.


In [ ]:
def train_and_score(sequences, labels, k=4, n_train=None, seed=SEED):
    sequences = pd.Series(sequences).reset_index(drop=True)
    labels = np.asarray(labels)
    row_indices = np.arange(len(sequences))
    train_indices, test_indices = train_test_split(
        row_indices, test_size=0.25, random_state=seed, stratify=labels
    )
    if n_train is not None and n_train < len(train_indices):
        train_indices, _ = train_test_split(
            train_indices, train_size=n_train, random_state=seed,
            stratify=labels[train_indices]
        )
    if n_train is not None and n_train > len(train_indices):
        raise ValueError("Requested training rows exceed the available training pool.")

    counter = CountVectorizer(analyzer="char", ngram_range=(k, k), lowercase=False)
    features_train = counter.fit_transform(sequences.iloc[train_indices])
    features_test = counter.transform(sequences.iloc[test_indices])
    model = LogisticRegression(solver="liblinear", max_iter=2000, random_state=seed)
    model.fit(features_train, labels[train_indices])
    scores = model.predict_proba(features_test)[:, 1]
    return roc_auc_score(labels[test_indices], scores)

print("Ready to repeat the same experiment with one setting changed.")


Finish the k-mer experiment with k = 4, 5, and 6. Each run prints the updated
table so you can see it arrive. The k = 4 row should reproduce the baseline.

**Try it.** Change one k value, rerun, and record the difference. If performance
levels off or drops at k = 6, propose an explanation: there are 4,096 possible
6-mers, but just 76 overlapping 6-mer positions in an 81-letter sequence.
Do not assume a drop must occur; describe your actual output.


In [ ]:
for k_value in [4, 5, 6]:
    auc_k = train_and_score(promoters["segment"], labels, k=k_value, seed=SEED)
    results.loc[len(results)] = ["k-mer length", f"k={k_value}", auc_k, len(labels_train), len(labels_test)]
    display(results.round({"auc": 3}))


### Experiment B — how many training examples?

Keep k fixed and reuse exactly the same test rows. We choose four training
sizes from the available training pool: about **10%, 25%, 50%, and 100%**.
The ceiling is calculated at runtime after the organism filter and holdout;
the old fixed values would exceed the available rows.

The next cell records the actual counts and scores. Look for diminishing
returns, if present. **Try it.** Change one fraction, keeping it above zero
and at or below one. Which mattered more in your runs: k or training size?


In [ ]:
max_train = len(labels_train)
sizes = []
for fraction in [0.1, 0.25, 0.5, 1.0]:
    sizes.append(max(2, int(max_train * fraction)))
print("Available training rows:", max_train)
print("Training sizes to compare:", sizes)

for training_size in sizes:
    auc_size = train_and_score(
        promoters["segment"], labels, k=K, n_train=training_size, seed=SEED
    )
    results.loc[len(results)] = ["training size", f"n_train={training_size}", auc_size, training_size, len(labels_test)]
    display(results.round({"auc": 3}))


The next figure puts the two experiments side by side. Each point is one run,
not an average across resamples. Compare the patterns while remembering that
we have looked at the test set repeatedly: it is now helping us explore choices,
so it is not an untouched final evaluation. If you reran a setting, the latest
row for that setting is shown.


In [ ]:
k_results = results[results["experiment"] == "k-mer length"].drop_duplicates("setting", keep="last").copy()
k_results["k"] = k_results["setting"].str.replace("k=", "", regex=False).astype(int)
k_results = k_results.sort_values("k")
size_results = results[results["experiment"] == "training size"].drop_duplicates("n_train", keep="last").sort_values("n_train")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(k_results["k"], k_results["auc"], "o-", color="#167b88")
axes[0].set(xlabel="k-mer length", ylabel="ROC-AUC", title="Change the representation")
axes[0].set_xticks(k_results["k"])
axes[1].plot(size_results["n_train"], size_results["auc"], "o-", color="#675aa0")
axes[1].set(xlabel="Training sequences", ylabel="ROC-AUC", title="Change the training size")
for ax in axes:
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig("figures/fig5_one_thing_at_a_time.png", dpi=150)
plt.show()
plt.close(fig)


### Experiment C — change only the seed

The *E. coli* positives, random negatives, k, and model settings stay fixed.
Changing the seed changes which rows land in training and test (and the
solver's random state). The next cell tries `SEED`, `SEED + 1`, and `SEED + 2`.

Look at the spread before interpreting tiny score differences. These three
runs are a sensitivity check, **not a confidence interval or significance test**.
Their spread describes this RND comparison; it cannot certify chance-level
performance for another negative class.

Changing `SEED` at the top and running everything is a broader experiment: it
also changes the online negative sample. With the bundled backup, the available
source pools are fixed, so that broader resampling has less scope.


In [ ]:
for split_seed in [SEED, SEED + 1, SEED + 2]:
    auc_seed = train_and_score(promoters["segment"], labels, k=K, seed=split_seed)
    results.loc[len(results)] = ["seed", f"seed={split_seed}", auc_seed, len(labels_train), len(labels_test)]
    display(results.round({"auc": 3}))


## 10 — What counts as a negative?

Everything so far used **random nucleotide sequence** as the negative class.
You have a working classifier and a strong first result on that comparison.

Now keep the model procedure unchanged: same k, seed, settings, positives,
class sizes, and positive train/test assignments. Change what the promoters
are compared against. Each stage fits a fresh model using the same procedure;
we are testing a data-design choice, not freezing the original fitted weights.

The next cell discovers the negative categories from the full available
source, not the filtered baseline. Descriptions explain the published
construction methods; an unknown future category is printed and triggers a
clear error rather than being silently assigned a meaning.


In [ ]:
negative_classes = sorted(promoters_all.loc[promoters_all["y"] == 0, "prom_class"].dropna().unique())
# These descriptions annotate observed categories; they do not select which ones exist.
negative_descriptions = {
    "RND": "random nucleotide sequence (zero-order generation)",
    "CDS": "real coding sequence sampled from sequenced genomes",
    "comp_kt1_kc3": "generated using a third-order Markov composition model",
}
for name in negative_classes:
    available = ((promoters_all["y"] == 0) & (promoters_all["prom_class"] == name)).sum()
    print(name, "-", negative_descriptions.get(name, "unrecognised category; inspect source"),
          "| available:", available)
if set(negative_classes) != set(negative_descriptions):
    raise ValueError("The negative categories differ from this lesson's verified source snapshot.")

# Discover the two alternatives from the observed values and their annotated meaning.
for name in negative_classes:
    if negative_descriptions[name].startswith("real coding"):
        coding_class = name
    if negative_descriptions[name].startswith("generated using a third-order"):
        composition_class = name


**Before you run the next cell. Write it down.**

- Which comparison will be **easiest**: random, random plus coding, or composition-matched?
- Which will be **hardest**?
- How large will the gap be — 0.05, 0.15, or 0.3?

Commit to an answer with your partner before running anything.

### Stage 1 — the anchor

Start with exactly the positives and RND negatives used all session. The next
cell should reproduce the baseline AUC. Positive rows keep their order, the
same shuffle and split are reused, and class sizes stay fixed. Watch the
single score arrive before moving on; it is our anchor for the next stages.


In [ ]:
# Stage 1 — same k, seed, positives, negative rows, and balanced class sizes.
stage1_data = pd.concat([positives, negatives], ignore_index=True)
stage1_data = stage1_data.sample(frac=1, random_state=SEED).reset_index(drop=True)
auc_rnd = train_and_score(stage1_data["segment"], stage1_data["y"], k=K, seed=SEED)
print("vs random nucleotides:      ", round(auc_rnd, 3))
print("Original baseline:          ", round(baseline_auc, 3))
results.loc[len(results)] = ["negative stage", "1: RND", auc_rnd, len(labels_train), len(labels_test)]
display(results.round({"auc": 3}))


### Stage 2 — bring in real coding DNA

Replace half of the random-negative row slots with coding-sequence examples.
The remaining random rows stay in exactly the same slots. The negative class
still has the same total size, and the positives have not moved. With an odd
class size, one extra negative remains random.

The next cell draws the coding pool, replaces the selected slots, refits the
same model procedure, and records its AUC. **Predict again:** does distinguishing
promoters from this mixed background sound like the same task as Stage 1?


In [ ]:
coding_pool = promoters_all[(promoters_all["y"] == 0) & (promoters_all["prom_class"] == coding_class)]
if len(coding_pool) < n_per_class:
    raise ValueError("Not enough coding negatives to keep the class sizes fixed.")
coding_negatives = coding_pool.sample(n=n_per_class, random_state=SEED)
coding_negatives = coding_negatives.sort_values("segment_id").reset_index(drop=True)

mixed_negatives = negatives.copy()
stage_rng = np.random.default_rng(SEED)
coding_slots = stage_rng.choice(n_per_class, size=n_per_class // 2, replace=False)
mixed_negatives.loc[coding_slots, :] = coding_negatives.loc[coding_slots, :]
stage2_data = pd.concat([positives, mixed_negatives], ignore_index=True)
stage2_data = stage2_data.sample(frac=1, random_state=SEED).reset_index(drop=True)
auc_rnd_cds = train_and_score(stage2_data["segment"], stage2_data["y"], k=K, seed=SEED)
print("vs random + coding sequence:", round(auc_rnd_cds, 3))
print("Negative mixture:")
print(mixed_negatives["prom_class"].value_counts())
results.loc[len(results)] = ["negative stage", "2: RND + CDS", auc_rnd_cds, len(labels_train), len(labels_test)]
display(results.round({"auc": 3}))


### Stage 3 — control more local sequence composition

Replace the negatives with the third-order Markov-generated set. This uses
local sequence statistics from the source promoter data, making it a stronger
composition control than independent random nucleotide generation.

The next cell keeps the same positives and class sizes and reports the third
score. A value near 0.5 would be compatible with little useful ranking signal;
an appreciably higher value means some signal remains. **Do not assume the old
multispecies result is the answer for this new E. coli comparison.**


In [ ]:
composition_pool = promoters_all[(promoters_all["y"] == 0) & (promoters_all["prom_class"] == composition_class)]
if len(composition_pool) < n_per_class:
    raise ValueError("Not enough composition negatives to keep the class sizes fixed.")
composition_negatives = composition_pool.sample(n=n_per_class, random_state=SEED)
composition_negatives = composition_negatives.sort_values("segment_id").reset_index(drop=True)
stage3_data = pd.concat([positives, composition_negatives], ignore_index=True)
stage3_data = stage3_data.sample(frac=1, random_state=SEED).reset_index(drop=True)
auc_comp = train_and_score(stage3_data["segment"], stage3_data["y"], k=K, seed=SEED)
print("vs composition-matched:     ", round(auc_comp, 3))
print("Drop from the random-negative comparison:", round(auc_rnd - auc_comp, 3))
results.loc[len(results)] = ["negative stage", "3: composition-matched", auc_comp, len(labels_train), len(labels_test)]
display(results.round({"auc": 3}))


### Figure 6 — three different questions, one model procedure

Now put the three scores together. The next cell draws the staged comparisons
in their teaching order, with a dashed **guessing** line at AUC = 0.5. All bars
use the same class sizes and positive rows. Read both the decline and the
distance of the final bar from chance; these answer different questions.


In [ ]:
stage_names = ["Random nucleotides", "Random + coding", "Composition-matched"]
stage_aucs = [auc_rnd, auc_rnd_cds, auc_comp]
fig, ax = plt.subplots(figsize=(9, 4.7))
bars = ax.bar(stage_names, stage_aucs, color=["#167b88", "#d47d36", "#675aa0"], width=0.65)
ax.bar_label(bars, fmt="%.3f", padding=5)
ax.axhline(0.5, color="gray", linestyle="--", label="Guessing (ROC-AUC = 0.5)")
ax.set(ylim=(0, 1.06), ylabel="ROC-AUC", xlabel="Negative examples",
       title=f"Same E. coli positives; {n_per_class:,} examples per label")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig("figures/fig6_negative_classes.png", dpi=150)
plt.show()
plt.close(fig)


### What just happened

Same k. Same seed. Same model settings and fitting procedure. Same promoters.
Same class sizes and positive train/test assignments. What changed was
**what those promoters were compared against**.

The default E. coli run shows a strong RND score, a lower mixed-background
score, and a further decline against composition-matched negatives. **The last
score is still clearly above 0.5. It is not a collapse to guessing.** Read your
actual three bars if you changed any settings.

This is not a broken model. It is a measurement—and the distinction between
“the score dropped” and “all signal disappeared” matters. The earlier value
near 0.48 came from a different, multispecies positive population. It cannot
be carried over after restricting the positives to E. coli.

### Why the composition control changes the task

A **third-order Markov model** generates the next base using the previous
three: `P(next base | previous three bases)`. Those transition probabilities
are closely related to 4-mer frequencies. The published method estimates
them using source promoter sequences and reverse complements, then samples
new sequences.

That controls aspects of **local word composition**, precisely the kind of
information our 4-mer representation can use. It does **not** make every
generated 81-base sequence carry exactly the same count vector as a positive,
and it does not mathematically guarantee an AUC of 0.5.

There is another important mismatch: our positives are E. coli-only, while
the composition-negative pool is not verified as matched to those selected
positives. Species-related composition may therefore remain useful. The missing
negative-species metadata prevents us from removing that confound here.

> **Changing the negative class changed the apparent success of the model.
> Stronger composition controls removed some predictive advantage; this
> E. coli experiment does not show that they removed all of it.**

The AT-fraction check is another view of the same fitted model, not independent
proof that all its signal was AT richness. Word composition includes much more
than the overall fraction of A and T.

### The design lesson

**Choosing the negative class is not just preparation. It is part of designing
the experiment.** “Promoter versus random sequence” and “promoter versus a
harder biological background” are different prediction problems.

When a paper reports a promoter classifier at 0.95, ask first:
**Promoters from where, compared against what, and evaluated with which split?**
Architecture comes after establishing what was actually measured.

### And what would help us go further?

Counting discards motif positions and spacing. A representation that can use
**where** motifs sit and **how they are arranged**—including the spacing of
the −35/−10 regions—lets us test an additional biological hypothesis.

It is a next experiment, not a guaranteed win. We also need background controls
matched to the intended organism and application, and an evaluation split
that tests the intended generalisation. That is the next rung of the ladder.

## 11 — Everything you did today 

The next cell displays and saves the complete record. There should be at least
twelve rows. Compare the three staged rows with one another; the first also
reproduces the baseline. Write down the comparison that changed what the
initial high score meant to you. `results.csv` preserves all scores.


In [ ]:
results.to_csv("results.csv", index=False)
print("Recorded experiments:", len(results))
display(results.round({"auc": 3}))


Three Design Sheet questions remain open:

- **The split:** can related sequences cross the holdout, and would this
  performance transfer to unseen species? Session 4 returns to that question.
- **The negative definition:** what background would make sense for the
  actual application? The global negative pools are not verified species
  matches to our E. coli positives. We measured their effect, not solved it.
- **What is being learned instead?** Species composition, general local word
  frequencies, and promoter motifs remain possible contributors. Position
  and spacing were discarded; coefficients alone do not resolve mechanism.

Return to the **Claim of the Week** vote from the opening lecture:

> “A promoter classifier with ROC-AUC 0.97 is ready for use on new bacterial species.”

Would you cast the same vote now? State which missing information would change
your answer: the negatives, the positive population, or the evaluation split.

## 12 -Break it

**in pairs.** Find the single change that most degrades the AUC.
Anything is allowed except breaking the code: a `NameError` is not a finding.

Report three things:

1. **The change** — one thing, described in a sentence.
2. **The numbers** — AUC before and AUC after.
3. **The mechanism** — why it dropped. This is the part that counts.

**Predict before you run.** Write down the direction and rough size first.
Possible starting points: make the negatives harder; shrink training size;
shorten sequences; corrupt some training labels; or, as an extension, train
on one species and test on another. Missing species names make that last option
less straightforward than a simple filter, especially for synthetic negatives.

The next cell is complete working code. Its default settings reproduce the
baseline. Change **one** marked constant and rerun. `BREAK_TRAIN_ROWS` changes
training size with the same test set; `BREAK_LENGTH` crops every sequence from
its start; `BREAK_LABEL_NOISE` flips the requested fraction of training labels.
Test labels stay untouched so a drop has an interpretable meaning.

The row is added to the same table and the CSV is updated. Your earlier RND baseline
model and scores remain available for comparison.


In [ ]:
BREAK_K = K                  # CHANGE ONE: try 2 or 6.
BREAK_TRAIN_ROWS = len(labels_train)  # CHANGE ONE: try 500; this is the maximum.
BREAK_LENGTH = 81            # CHANGE ONE: try 30; keep at least BREAK_K.
BREAK_LABEL_NOISE = 0.0      # CHANGE ONE: try 0.25 or 0.50; fraction to flip.

assert 1 <= BREAK_K <= BREAK_LENGTH <= 81
assert 0 <= BREAK_LABEL_NOISE <= 1

break_sequences = promoters["segment"].str[:BREAK_LENGTH]
break_labels = promoters["y"].to_numpy()
break_indices = np.arange(len(promoters))
break_train_indices, break_test_indices = train_test_split(
    break_indices, test_size=0.25, random_state=SEED, stratify=break_labels
)
assert 10 <= BREAK_TRAIN_ROWS <= len(break_train_indices)
if BREAK_TRAIN_ROWS < len(break_train_indices):
    break_train_indices, _ = train_test_split(
        break_train_indices, train_size=BREAK_TRAIN_ROWS, random_state=SEED,
        stratify=break_labels[break_train_indices]
    )

# This copy is the only label array we corrupt. Held-out answers stay correct.
break_training_labels = break_labels[break_train_indices].copy()
break_rng = np.random.default_rng(SEED)
number_to_flip = int(len(break_training_labels) * BREAK_LABEL_NOISE)
flip_indices = break_rng.choice(len(break_training_labels), size=number_to_flip, replace=False)
break_training_labels[flip_indices] = 1 - break_training_labels[flip_indices]

break_counter = CountVectorizer(analyzer="char", ngram_range=(BREAK_K, BREAK_K), lowercase=False)
break_features_train = break_counter.fit_transform(break_sequences.iloc[break_train_indices])
break_features_test = break_counter.transform(break_sequences.iloc[break_test_indices])
break_model = LogisticRegression(solver="liblinear", max_iter=2000, random_state=SEED)
break_model.fit(break_features_train, break_training_labels)
break_scores = break_model.predict_proba(break_features_test)[:, 1]
break_auc = roc_auc_score(break_labels[break_test_indices], break_scores)

break_setting = f"k={BREAK_K}; train={BREAK_TRAIN_ROWS}; length={BREAK_LENGTH}; noise={BREAK_LABEL_NOISE}"
results.loc[len(results)] = ["break it", break_setting, break_auc, len(break_train_indices), len(break_test_indices)]
print(f"Baseline AUC: {baseline_auc:.3f} | Changed AUC: {break_auc:.3f}")
print(f"AUC drop: {baseline_auc - break_auc:.3f}")
results.to_csv("results.csv", index=False)
display(results.round({"auc": 3}))


### Your report — edit this table

| Item | Your answer |
|---|---|
| One change | |
| Prediction before running | |
| AUC before | |
| AUC after | |
| Proposed mechanism | |

Report back in **three minutes, no slides**. Explain the mechanism, not just
which number was smallest.

## 13 — Stretch · optional, unmarked

1. Evaluate on `test_multispecies` instead of the random split. What happens?
   Note that both the positive population and negative mixture may differ.
2. Replace `CountVectorizer` counts with TF-IDF. Does it help?
3. Add positional features: GC content in the first 30 versus last 30 letters.
   Does the model use them?

These are optional extensions, not missing steps in the working notebook.
The first needs the Hub's official split and previews Session 4. 

### Sources

- [Dataset card and licence](https://huggingface.co/datasets/neuralbioinfo/bacterial_promoters).
- [ProkBERT: promoter benchmark construction and methods](https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2023.1331233/full).
- [scikit-learn logistic regression reference](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

The benchmark's negative construction and our random split define what the
score means. A promoter-like score does not establish toxin expression.
